In [4]:
import os
import torch
import tqdm
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

import articulate as art

from config import *
from utils.model_utils import load_model
from data import PoseDataset
import matplotlib
matplotlib.use("Agg")  # no GUI / no OpenGL

# -------------------------
# PoseEvaluator (same as before)
# -------------------------
class PoseEvaluator:
    def __init__(self):
        fps = getattr(datasets, "fps", 60)
        self._eval_fn = art.FullMotionEvaluator(
            paths.smpl_file,
            joint_mask=torch.tensor([2, 5, 16, 20]),
            fps=fps
        )
        try:
            self._ignored = list(joint_set.ignored)
        except Exception:
            self._ignored = []

    def eval(self, pose_p, pose_t, tran_p=None, tran_t=None):
        pose_p = pose_p.clone().view(-1, 24, 3, 3)
        pose_t = pose_t.clone().view(-1, 24, 3, 3)

        if tran_p is None:
            tran_p = torch.zeros((pose_p.shape[0], 3), device=pose_p.device)
        if tran_t is None:
            tran_t = torch.zeros((pose_t.shape[0], 3), device=pose_t.device)

        tran_p = tran_p.clone().view(-1, 3)
        tran_t = tran_t.clone().view(-1, 3)

        if len(self._ignored) > 0:
            pose_p[:, self._ignored] = torch.eye(3, device=pose_p.device)
            pose_t[:, self._ignored] = torch.eye(3, device=pose_t.device)

        errs = self._eval_fn(pose_p, pose_t, tran_p=tran_p, tran_t=tran_t)

        def to_scalar(x: torch.Tensor) -> torch.Tensor:
            # Some metrics are returned per-frame; collapse them to one number
            return x.mean() if x.numel() > 1 else x

        metrics = [
            to_scalar(errs[9]),        # SIP Error (deg)
            to_scalar(errs[3]),        # Angular Error (deg)
            to_scalar(errs[9]),        # Masked Angular Error (deg) (as in your code)
            to_scalar(errs[0]) * 100,  # Positional Error (cm)
            to_scalar(errs[7]) * 100,  # Masked Positional Error (cm)
            to_scalar(errs[1]) * 100,  # Mesh Error (cm)
            to_scalar(errs[4]) / 100,  # Jitter Error (100m/s^3)
            to_scalar(errs[6]),        # Distance Error (cm)
        ]
        return torch.stack(metrics)

    @staticmethod
    def print(mean_std_8x2):
        names = [
            'SIP Error (deg)',
            'Angular Error (deg)',
            'Masked Angular Error (deg)',
            'Positional Error (cm)',
            'Masked Positional Error (cm)',
            'Mesh Error (cm)',
            'Jitter Error (100m/s^3)',
            'Distance Error (cm)'
        ]
        for i, name in enumerate(names):
            m = mean_std_8x2[i, 0].item()
            s = mean_std_8x2[i, 1].item()
            print(f"{name}: {m:.2f} (+/- {s:.2f})")



# -------------------------
# Qualitative GIF from JOINTS (no SMPL needed)
# -------------------------
# Simple chain for 24 SMPL joints (works if joints are 24). If your joints count differs,
# we’ll auto-fallback to scatter-only.
SMPL_PARENTS = [-1, 0, 0, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 9, 9, 12, 13, 14, 16, 17, 18, 19, 20, 21]


def _draw_joints(ax, joints, title=""):
    """
    joints: (J,3) torch or np
    draws scatter + (optional) bones if J==24
    """
    if torch.is_tensor(joints):
        joints = joints.detach().cpu().numpy()

    ax.cla()
    ax.set_title(title)
    ax.set_xlabel("X"); ax.set_ylabel("Y"); ax.set_zlabel("Z")
    ax.scatter(joints[:, 0], joints[:, 1], joints[:, 2], s=10)

    J = joints.shape[0]
    if J == 24:
        for j, p in enumerate(SMPL_PARENTS):
            if p >= 0:
                ax.plot([joints[p, 0], joints[j, 0]],
                        [joints[p, 1], joints[j, 1]],
                        [joints[p, 2], joints[j, 2]], linewidth=2)


def save_gt_vs_pred_gif_from_joints(
    joints_gt, joints_pred,
    out_path,
    fps=60,
    max_frames=300,
    stride=5
):
    import matplotlib
    matplotlib.use("Agg")  # extra safety if called in notebook

    if torch.is_tensor(joints_gt):
        joints_gt = joints_gt.detach().cpu()
    if torch.is_tensor(joints_pred):
        joints_pred = joints_pred.detach().cpu()

    T = min(joints_gt.shape[0], joints_pred.shape[0], max_frames)

    fig = plt.figure(figsize=(10, 5), dpi=100)
    ax1 = fig.add_subplot(121, projection="3d")
    ax2 = fig.add_subplot(122, projection="3d")

    # Use a range iterator (no big list allocation)
    frame_indices = range(0, T, stride)

    def update(i):
        _draw_joints(ax1, joints_gt[i], "Ground Truth")
        _draw_joints(ax2, joints_pred[i], "Prediction")
        return []

    anim = FuncAnimation(
        fig,
        update,
        frames=frame_indices,
        interval=1000 / max(1, (fps / stride)),
        blit=False
    )

    # GIF writing is the heavy part — keep fps low
    anim.save(out_path, writer="pillow", fps=max(5, fps // stride))
    plt.close(fig)
    return out_path




# -------------------------
# Main: run exactly like you want
# -------------------------
@torch.no_grad()
def evaluate_pose_with_qualitative(
    model,
    dataset,
    save_video=True,
    video_seq_id=0,
    out_dir="qualitative",
    max_video_frames=1200
):
    device = getattr(model_config, "device", torch.device("cuda" if torch.cuda.is_available() else "cpu"))
    fps = getattr(datasets, "fps", 60)

    evaluator = PoseEvaluator()
    model.eval()

    os.makedirs(out_dir, exist_ok=True)

    all_errs = []

    for seq_idx, (imu, pose_r6d, joint_gt, tran_gt) in enumerate(tqdm.tqdm(dataset, desc="Evaluating")):
        imu = imu.to(device)
        pose_r6d = pose_r6d.to(device)
        tran_gt = tran_gt.to(device)

        # joint_gt might already be (T,J,3) on CPU; move only if needed for indexing
        # We'll keep it on CPU for visualization
        if torch.is_tensor(joint_gt) and joint_gt.device.type != "cpu":
            joint_gt_cpu = joint_gt.detach().cpu()
        else:
            joint_gt_cpu = joint_gt

        model.reset()
        pose_p, joint_p, tran_p, _ = model.forward_offline(imu.unsqueeze(0), [imu.shape[0]])

        # Convert GT pose to rot matrices for metric evaluation
        pose_t = art.math.r6d_to_rotation_matrix(pose_r6d)

        # Quantitative GT metrics
        all_errs.append(evaluator.eval(pose_p, pose_t, tran_p=tran_p, tran_t=tran_gt))

        # Qualitative video (use joints directly)
        if save_video and seq_idx == video_seq_id:
            # joint_p expected shape (T,J,3)
            if torch.is_tensor(joint_p) and joint_p.device.type != "cpu":
                joint_p_cpu = joint_p.detach().cpu()
            else:
                joint_p_cpu = joint_p

            out_path = os.path.join(out_dir, f"gt_vs_pred_seq{video_seq_id}.gif")
            saved = save_gt_vs_pred_gif_from_joints(
                joint_gt_cpu, joint_p_cpu,
                out_path,
                fps=fps,
                max_frames=max_video_frames
            )
            print(f"[Saved qualitative GIF] {saved}")

    # Print mean+std
    errs = torch.stack(all_errs)  # (N,8)
    mean = errs.mean(dim=0)
    std = errs.std(dim=0)

    print("\n===== Pose Evaluation (GT-based) =====")
    evaluator.print(torch.stack([mean, std], dim=1))

    return {"mean": mean.detach().cpu(), "std": std.detach().cpu()}


In [5]:
model_path = r"C:\deepika\2.mobileposer\MobilePoser\mobileposer\deepika\model_finetuned.pth"
dataset_name = "dip"

model = load_model(model_path)

if dataset_name not in datasets.test_datasets:
    raise ValueError(f"Test dataset: {dataset_name} not found.")

dataset = PoseDataset(fold="test", evaluate=dataset_name)

print(f"Starting evaluation: {dataset_name.capitalize()}")

results = evaluate_pose_with_qualitative(
    model,
    dataset,
    save_video=False,
    video_seq_id=0,
    out_dir="qualitative",
    max_video_frames=300   # must match max_frames idea
)



100%|██████████| 1/1 [00:00<00:00,  6.67it/s]


Starting evaluation: Dip


Evaluating: 100%|██████████| 228/228 [05:25<00:00,  1.43s/it]


===== Pose Evaluation (GT-based) =====
SIP Error (deg): 19.52 (+/- 7.44)
Angular Error (deg): 17.56 (+/- 6.10)
Masked Angular Error (deg): 19.52 (+/- 7.44)
Positional Error (cm): 7.20 (+/- 2.60)
Masked Positional Error (cm): 6.65 (+/- 2.58)
Mesh Error (cm): 9.02 (+/- 3.48)
Jitter Error (100m/s^3): 0.29 (+/- 0.26)
Distance Error (cm): 6.63 (+/- 0.22)


In [ ]:
from IPython.display import Image
Image("qualitative/gt_vs_pred_seq0.gif")

